# 🏥 VLM Knowledge Distillation on VQA-RAD
## Teacher: CLIP-ViT-L/14 → Student: MobileNetV2

**Pipeline Overview:**
1. Load & preprocess VQA-RAD dataset
2. Train/Test/Validation splits
3. Baseline accuracy — Teacher (CLIP-ViT-L/14)
4. Baseline accuracy — Student (MobileNetV2, no KD)
5. Knowledge Distillation training
6. Post-KD Student accuracy
7. Sample visualization & comparison


## 1. Install Dependencies

In [ ]:
# Install required packages
!pip install -q transformers datasets torch torchvision open_clip_torch Pillow matplotlib seaborn scikit-learn tqdm kaggle
print("✅ All dependencies installed.")

## 2. Imports & Configuration

In [ ]:
import os
import json
import random
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
from PIL import Image
from pathlib import Path
from tqdm import tqdm
from collections import Counter

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader, random_split
import torchvision.models as models
import torchvision.transforms as transforms

import open_clip
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
from sklearn.preprocessing import LabelEncoder

# Reproducibility
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"🖥️  Device: {DEVICE}")
if torch.cuda.is_available():
    print(f"   GPU: {torch.cuda.get_device_name(0)}")

## 3. Download VQA-RAD Dataset from Kaggle

In [ ]:
# ─────────────────────────────────────────────────────────────
# Option A: Kaggle API (recommended)
# Place your kaggle.json in ~/.kaggle/ before running
# ─────────────────────────────────────────────────────────────
import os

KAGGLE_DIR = os.path.expanduser('~/.kaggle')
os.makedirs(KAGGLE_DIR, exist_ok=True)

# If running on Kaggle itself, data is already mounted:
KAGGLE_INPUT = '/kaggle/input/vqa-rad-visual-question-answering-radiology'

if os.path.exists(KAGGLE_INPUT):
    DATA_DIR = KAGGLE_INPUT
    print(f"✅ Using Kaggle input: {DATA_DIR}")
else:
    # Download via API
    !kaggle datasets download -d shashankshekhar1205/vqa-rad-visual-question-answering-radiology --unzip -p ./vqa_rad_data
    DATA_DIR = './vqa_rad_data'
    print(f"✅ Downloaded to: {DATA_DIR}")

print("\n📁 Dataset contents:")
for root, dirs, files in os.walk(DATA_DIR):
    level = root.replace(DATA_DIR, '').count(os.sep)
    indent = ' ' * 2 * level
    print(f"{indent}{os.path.basename(root)}/")
    subindent = ' ' * 2 * (level + 1)
    for f in files[:10]:
        print(f"{subindent}{f}")
    if len(files) > 10:
        print(f"{subindent}... and {len(files)-10} more files")

## 4. Load & Explore the Dataset

In [ ]:
def find_json_files(base_dir):
    """Locate all JSON/JSONL annotation files."""
    json_files = []
    for root, _, files in os.walk(base_dir):
        for f in files:
            if f.endswith('.json') or f.endswith('.jsonl'):
                json_files.append(os.path.join(root, f))
    return json_files

def find_image_dir(base_dir):
    """Find directory containing images."""
    for root, dirs, files in os.walk(base_dir):
        imgs = [f for f in files if f.lower().endswith(('.jpg','.jpeg','.png'))]
        if len(imgs) > 10:
            return root, imgs
    return base_dir, []

json_files = find_json_files(DATA_DIR)
print("JSON files found:", json_files)

img_dir, img_list = find_image_dir(DATA_DIR)
print(f"Image directory: {img_dir}")
print(f"Total images found: {len(img_list)}")

In [ ]:
# Load annotation file
def load_annotations(json_files):
    all_data = []
    for jf in json_files:
        try:
            with open(jf, 'r') as f:
                content = f.read().strip()
                if content.startswith('['):
                    data = json.loads(content)
                else:
                    data = [json.loads(line) for line in content.split('\n') if line.strip()]
            all_data.extend(data)
            print(f"  Loaded {len(data)} samples from {os.path.basename(jf)}")
        except Exception as e:
            print(f"  ⚠️ Could not load {jf}: {e}")
    return all_data

raw_data = load_annotations(json_files)
print(f"\n✅ Total QA pairs loaded: {len(raw_data)}")
if raw_data:
    print("\nSample entry keys:", list(raw_data[0].keys()))
    print("\nSample entry:")
    print(json.dumps(raw_data[0], indent=2))

In [ ]:
# Build a clean DataFrame
def build_dataframe(raw_data, img_dir):
    rows = []
    for item in raw_data:
        # Handle various VQA-RAD key conventions
        img_name = item.get('image_name', item.get('image', item.get('img_id', '')))
        question = item.get('question', item.get('Question', ''))
        answer = item.get('answer', item.get('Answer', ''))
        qtype = item.get('question_type', item.get('answer_type', 'unknown'))
        phrase_type = item.get('phrase_type', '')
        
        # Resolve image path
        img_path = None
        for ext in ['', '.jpg', '.jpeg', '.png']:
            candidate = os.path.join(img_dir, str(img_name) + ext)
            if os.path.exists(candidate):
                img_path = candidate
                break
        
        rows.append({
            'image_name': str(img_name),
            'image_path': img_path,
            'question': str(question),
            'answer': str(answer).strip().lower(),
            'question_type': str(qtype).lower(),
            'phrase_type': str(phrase_type).lower()
        })
    
    df = pd.DataFrame(rows)
    df = df[df['image_path'].notna()].reset_index(drop=True)
    return df

df = build_dataframe(raw_data, img_dir)
print(f"DataFrame shape: {df.shape}")
df.head()

In [ ]:
print("=" * 55)
print("📊 DATASET STATISTICS")
print("=" * 55)
print(f"Total QA pairs:      {len(df)}")
print(f"Unique images:       {df['image_name'].nunique()}")
print(f"Unique answers:      {df['answer'].nunique()}")
print(f"\nQuestion types:\n{df['question_type'].value_counts()}")
print(f"\nTop 20 answers:")
print(df['answer'].value_counts().head(20))

## 5. Visualize Sample Images + QA Pairs

In [ ]:
def show_samples(df, n=8, title="VQA-RAD Sample QA Pairs"):
    sample = df.dropna(subset=['image_path']).sample(min(n, len(df)), random_state=SEED)
    cols = 4
    rows = (len(sample) + cols - 1) // cols
    fig, axes = plt.subplots(rows, cols, figsize=(20, rows * 5))
    axes = axes.flatten()
    
    for i, (_, row) in enumerate(sample.iterrows()):
        ax = axes[i]
        try:
            img = Image.open(row['image_path']).convert('RGB')
            ax.imshow(img, cmap='gray')
        except:
            ax.set_facecolor('#1a1a2e')
            ax.text(0.5, 0.5, 'Image\nNot Found', ha='center', va='center',
                    color='white', fontsize=12, transform=ax.transAxes)
        
        q_wrap = '\n'.join([row['question'][j:j+35] for j in range(0, min(len(row['question']),105), 35)])
        ax.set_title(f"Q: {q_wrap}\n✅ A: {row['answer']}",
                     fontsize=9, pad=4, color='#1a1a2e',
                     bbox=dict(boxstyle='round,pad=0.3', facecolor='#e8f4fd', alpha=0.9))
        ax.axis('off')
    
    for j in range(i + 1, len(axes)):
        axes[j].axis('off')
    
    fig.suptitle(title, fontsize=16, fontweight='bold', y=1.02)
    plt.tight_layout()
    plt.savefig('vqa_rad_samples.png', dpi=120, bbox_inches='tight')
    plt.show()
    print("✅ Saved: vqa_rad_samples.png")

show_samples(df, n=8)

In [ ]:
# Dataset Distribution Plots
fig, axes = plt.subplots(1, 3, figsize=(20, 5))

# Answer frequency
top_ans = df['answer'].value_counts().head(20)
axes[0].barh(top_ans.index[::-1], top_ans.values[::-1], color='#4e9af1')
axes[0].set_title('Top 20 Answers', fontsize=13, fontweight='bold')
axes[0].set_xlabel('Count')

# Question types
qt = df['question_type'].value_counts()
axes[1].pie(qt.values, labels=qt.index, autopct='%1.1f%%',
            colors=plt.cm.Set3.colors, startangle=90)
axes[1].set_title('Question Types', fontsize=13, fontweight='bold')

# Answer length distribution
df['answer_len'] = df['answer'].str.split().str.len()
axes[2].hist(df['answer_len'], bins=20, color='#f18b4e', edgecolor='white', linewidth=0.5)
axes[2].set_title('Answer Length Distribution (words)', fontsize=13, fontweight='bold')
axes[2].set_xlabel('Words in Answer')
axes[2].set_ylabel('Frequency')

plt.tight_layout()
plt.savefig('dataset_distribution.png', dpi=120, bbox_inches='tight')
plt.show()

## 6. Prepare Labels & Train/Val/Test Splits

In [ ]:
# ── Step 1: pick top-N answers ───────────────────────────────
TOP_N_ANSWERS = 50  # Reduce for faster training; increase for full study

top_answers = df['answer'].value_counts().head(TOP_N_ANSWERS).index.tolist()
df_filtered = df[df['answer'].isin(top_answers)].copy().reset_index(drop=True)

# ── Step 2: drop classes with < MIN_SAMPLES_PER_CLASS samples ─
# Stratified splitting into 3 parts requires each class to have
# at least 3 samples (1 per split). We use 6 as a safe margin.
MIN_SAMPLES_PER_CLASS = 6
counts = df_filtered['answer'].value_counts()
valid_answers = counts[counts >= MIN_SAMPLES_PER_CLASS].index.tolist()
removed = len(counts) - len(valid_answers)
df_filtered = df_filtered[df_filtered['answer'].isin(valid_answers)].copy().reset_index(drop=True)
print(f"Removed {removed} rare classes with fewer than {MIN_SAMPLES_PER_CLASS} samples.")

# ── Step 3: encode labels ─────────────────────────────────────
le = LabelEncoder()
df_filtered['label'] = le.fit_transform(df_filtered['answer'])
NUM_CLASSES = len(le.classes_)

print(f"Filtered to top-{TOP_N_ANSWERS} answers (after rare-class removal)")
print(f"Remaining samples : {len(df_filtered)}")
print(f"Number of classes : {NUM_CLASSES}")
print(f"Classes (first 10): {list(le.classes_[:10])} ...")

# ── Step 4: 70 / 15 / 15 stratified split ────────────────────
from sklearn.model_selection import train_test_split

df_train, df_temp = train_test_split(
    df_filtered, test_size=0.30, random_state=SEED,
    stratify=df_filtered['label']
)

# Guard: if any class in df_temp has only 1 sample, fall back
# to non-stratified for the val/test step to avoid the crash.
temp_counts = df_temp['label'].value_counts()
if temp_counts.min() >= 2:
    df_val, df_test = train_test_split(
        df_temp, test_size=0.50, random_state=SEED,
        stratify=df_temp['label']
    )
    print("\n✅ Both splits used stratified sampling.")
else:
    df_val, df_test = train_test_split(
        df_temp, test_size=0.50, random_state=SEED
    )
    print("\n⚠️  Val/Test used random (non-stratified) — some classes had only 1 sample in df_temp.")

df_train = df_train.reset_index(drop=True)
df_val   = df_val.reset_index(drop=True)
df_test  = df_test.reset_index(drop=True)

print(f"\n📊 Final split sizes:")
print(f"   Train : {len(df_train):>5}  ({len(df_train)/len(df_filtered)*100:.1f}%)")
print(f"   Val   : {len(df_val):>5}  ({len(df_val)/len(df_filtered)*100:.1f}%)")
print(f"   Test  : {len(df_test):>5}  ({len(df_test)/len(df_filtered)*100:.1f}%)")

## 7. Dataset & DataLoader Classes

In [ ]:
# ── Image transforms ─────────────────────────────────────────
CLIP_SIZE   = 224   # CLIP-ViT-L native size
MOBILE_SIZE = 224   # MobileNetV2 standard size

clip_transform = transforms.Compose([
    transforms.Resize((CLIP_SIZE, CLIP_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.48145466, 0.4578275, 0.40821073],
                         std=[0.26862954, 0.26130258, 0.27577711])
])

mobile_transform_train = transforms.Compose([
    transforms.Resize((MOBILE_SIZE, MOBILE_SIZE)),
    transforms.RandomHorizontalFlip(),
    transforms.ColorJitter(brightness=0.2, contrast=0.2),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
])

mobile_transform_eval = transforms.Compose([
    transforms.Resize((MOBILE_SIZE, MOBILE_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
])


class VQARadDataset(Dataset):
    """
    VQA-RAD dataset loader.
    Returns both CLIP-preprocessed image and MobileNet-preprocessed image.
    """
    def __init__(self, df, clip_transform=None, mobile_transform=None):
        self.df = df.reset_index(drop=True)
        self.clip_transform   = clip_transform
        self.mobile_transform = mobile_transform
    
    def __len__(self):
        return len(self.df)
    
    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        try:
            img = Image.open(row['image_path']).convert('RGB')
        except:
            img = Image.fromarray(np.zeros((224, 224, 3), dtype=np.uint8))
        
        clip_img   = self.clip_transform(img)   if self.clip_transform   else img
        mobile_img = self.mobile_transform(img) if self.mobile_transform else img
        label      = int(row['label'])
        
        return {
            'clip_img':   clip_img,
            'mobile_img': mobile_img,
            'label':      torch.tensor(label, dtype=torch.long),
            'question':   row['question'],
            'answer':     row['answer']
        }


BATCH_SIZE = 32

train_ds = VQARadDataset(df_train, clip_transform, mobile_transform_train)
val_ds   = VQARadDataset(df_val,   clip_transform, mobile_transform_eval)
test_ds  = VQARadDataset(df_test,  clip_transform, mobile_transform_eval)

train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True,  num_workers=2, pin_memory=True)
val_loader   = DataLoader(val_ds,   batch_size=BATCH_SIZE, shuffle=False, num_workers=2, pin_memory=True)
test_loader  = DataLoader(test_ds,  batch_size=BATCH_SIZE, shuffle=False, num_workers=2, pin_memory=True)

print(f"✅ DataLoaders ready")
print(f"   Train batches : {len(train_loader)}")
print(f"   Val   batches : {len(val_loader)}")
print(f"   Test  batches : {len(test_loader)}")

## 8. Teacher Model — CLIP-ViT-L/14

In [ ]:
# ── Load CLIP-ViT-L/14 via open_clip ─────────────────────────
print("⏳ Loading CLIP-ViT-L/14 ...")
clip_model, _, clip_preprocess = open_clip.create_model_and_transforms(
    'ViT-L-14', pretrained='openai'
)
clip_model = clip_model.to(DEVICE)
clip_model.eval()

CLIP_EMBED_DIM = clip_model.visual.output_dim   # 768 for ViT-L/14
print(f"✅ CLIP-ViT-L/14 loaded. Visual embed dim = {CLIP_EMBED_DIM}")
total_params = sum(p.numel() for p in clip_model.parameters()) / 1e6
print(f"   Total parameters: {total_params:.1f}M")


# ── Teacher Classifier Head ───────────────────────────────────
class TeacherCLIP(nn.Module):
    """CLIP-ViT-L backbone + linear classifier head."""
    def __init__(self, clip_model, embed_dim, num_classes):
        super().__init__()
        self.backbone   = clip_model.visual
        self.classifier = nn.Sequential(
            nn.LayerNorm(embed_dim),
            nn.Dropout(0.1),
            nn.Linear(embed_dim, num_classes)
        )

    def forward(self, x):
        feats  = self.backbone(x).float()   # [B, 768]
        logits = self.classifier(feats)     # [B, num_classes]
        return logits, feats


teacher = TeacherCLIP(clip_model, CLIP_EMBED_DIM, NUM_CLASSES).to(DEVICE)

# ── Selective Unfreeze: last 4 transformer blocks + head ──────
# Step 1 — freeze everything
for param in teacher.backbone.parameters():
    param.requires_grad = False

# Step 2 — unfreeze last 4 transformer blocks
UNFREEZE_LAST_N = 4
all_blocks = list(teacher.backbone.transformer.resblocks)
for block in all_blocks[-UNFREEZE_LAST_N:]:
    for param in block.parameters():
        param.requires_grad = True

# Step 3 — unfreeze final layer norm and projection
for param in teacher.backbone.ln_post.parameters():
    param.requires_grad = True

if hasattr(teacher.backbone, 'proj') and teacher.backbone.proj is not None:
    teacher.backbone.proj.requires_grad = True

# Step 4 — classifier head is always trainable
for param in teacher.classifier.parameters():
    param.requires_grad = True

frozen    = sum(p.numel() for p in teacher.backbone.parameters() if not p.requires_grad) / 1e6
trainable = sum(p.numel() for p in teacher.parameters()          if     p.requires_grad) / 1e6
print(f"\n✅ TeacherCLIP ready.")
print(f"   Frozen backbone params    : {frozen:.1f}M")
print(f"   Trainable params (total)  : {trainable:.2f}M  (last {UNFREEZE_LAST_N} blocks + LN + proj + head)")

In [ ]:
TEACHER_EPOCHS = 10   # was 10, more epochs needed for fine-tuning

# Separate LRs: very small for backbone, normal for head
backbone_params    = [p for n, p in teacher.backbone.named_parameters() if p.requires_grad]
classifier_params  = list(teacher.classifier.parameters())

teacher_optimizer = torch.optim.Adam([
    {'params': backbone_params,   'lr': 1e-5},   # small LR — don't destroy pretrained weights
    {'params': classifier_params, 'lr': 1e-3},   # normal LR for fresh head
], weight_decay=1e-4)

scheduler_t = torch.optim.lr_scheduler.CosineAnnealingLR(
    teacher_optimizer, T_max=TEACHER_EPOCHS
)
criterion = nn.CrossEntropyLoss()

# ── rest of the training loop is unchanged ────────────────────
teacher_train_losses, teacher_val_losses = [], []
teacher_train_accs,   teacher_val_accs   = [], []

print("🎓 Training Teacher (CLIP-ViT-L/14) with partial unfreeze ...")
best_val_acc = 0

for epoch in range(1, TEACHER_EPOCHS + 1):
    tr_loss, tr_acc = train_one_epoch(teacher, train_loader, teacher_optimizer,
                                       criterion, DEVICE, use_clip_img=True)
    vl_loss, vl_acc, _, _ = evaluate(teacher, val_loader, criterion, DEVICE, use_clip_img=True)
    scheduler_t.step()

    teacher_train_losses.append(tr_loss)
    teacher_val_losses.append(vl_loss)
    teacher_train_accs.append(tr_acc)
    teacher_val_accs.append(vl_acc)

    if vl_acc > best_val_acc:
        best_val_acc = vl_acc
        torch.save(teacher.state_dict(), 'best_teacher.pth')

    print(f"  Epoch [{epoch:02d}/{TEACHER_EPOCHS}]  "
          f"Train Loss: {tr_loss:.4f}  Train Acc: {tr_acc*100:.2f}%  "
          f"Val Acc: {vl_acc*100:.2f}%")

print(f"\n✅ Teacher best val accuracy: {best_val_acc*100:.2f}%")

In [ ]:
# ── Teacher Classifier Head ───────────────────────────────────
# Freeze CLIP backbone; add a linear classification head

class TeacherCLIP(nn.Module):
    """CLIP-ViT-L backbone + linear classifier head."""
    def __init__(self, clip_model, embed_dim, num_classes):
        super().__init__()
        self.backbone = clip_model.visual
        self.classifier = nn.Sequential(
            nn.LayerNorm(embed_dim),
            nn.Dropout(0.1),
            nn.Linear(embed_dim, num_classes)
        )
    
    def forward(self, x):
        with torch.no_grad():
            feats = self.backbone(x).float()   # [B, 768]
        logits = self.classifier(feats)        # [B, num_classes]
        return logits, feats

teacher = TeacherCLIP(clip_model, CLIP_EMBED_DIM, NUM_CLASSES).to(DEVICE)

# Only train the classification head
for p in teacher.backbone.parameters():
    p.requires_grad = False

trainable = sum(p.numel() for p in teacher.parameters() if p.requires_grad)
print(f"✅ TeacherCLIP ready. Trainable params: {trainable/1e6:.2f}M (head only)")

### 8.1 Train Teacher Classifier Head

In [ ]:
def train_one_epoch(model, loader, optimizer, criterion, device, use_clip_img=True):
    model.train()
    total_loss, correct, total = 0, 0, 0
    
    for batch in tqdm(loader, leave=False):
        imgs   = batch['clip_img' if use_clip_img else 'mobile_img'].to(device)
        labels = batch['label'].to(device)
        
        optimizer.zero_grad()
        output = model(imgs)
        logits = output[0] if isinstance(output, tuple) else output
        
        loss = criterion(logits, labels)
        loss.backward()
        optimizer.step()
        
        total_loss += loss.item() * labels.size(0)
        correct    += (logits.argmax(1) == labels).sum().item()
        total      += labels.size(0)
    
    return total_loss / total, correct / total


@torch.no_grad()
def evaluate(model, loader, criterion, device, use_clip_img=True):
    model.eval()
    total_loss, correct, total = 0, 0, 0
    all_preds, all_labels = [], []
    
    for batch in loader:
        imgs   = batch['clip_img' if use_clip_img else 'mobile_img'].to(device)
        labels = batch['label'].to(device)
        
        output = model(imgs)
        logits = output[0] if isinstance(output, tuple) else output
        
        loss = criterion(logits, labels)
        total_loss += loss.item() * labels.size(0)
        preds       = logits.argmax(1)
        correct    += (preds == labels).sum().item()
        total      += labels.size(0)
        all_preds.extend(preds.cpu().tolist())
        all_labels.extend(labels.cpu().tolist())
    
    return total_loss / total, correct / total, all_preds, all_labels


# ── Train Teacher ─────────────────────────────────────────────
TEACHER_EPOCHS = 100
teacher_optimizer = torch.optim.Adam(teacher.classifier.parameters(), lr=1e-3)
scheduler_t = torch.optim.lr_scheduler.CosineAnnealingLR(teacher_optimizer, T_max=TEACHER_EPOCHS)
criterion   = nn.CrossEntropyLoss()

teacher_train_losses, teacher_val_losses = [], []
teacher_train_accs,   teacher_val_accs   = [], []

print("🎓 Training Teacher (CLIP-ViT-L/14) classifier head ...")
best_val_acc = 0

for epoch in range(1, TEACHER_EPOCHS + 1):
    tr_loss, tr_acc = train_one_epoch(teacher, train_loader, teacher_optimizer,
                                       criterion, DEVICE, use_clip_img=True)
    vl_loss, vl_acc, _, _ = evaluate(teacher, val_loader, criterion, DEVICE, use_clip_img=True)
    scheduler_t.step()
    
    teacher_train_losses.append(tr_loss)
    teacher_val_losses.append(vl_loss)
    teacher_train_accs.append(tr_acc)
    teacher_val_accs.append(vl_acc)
    
    if vl_acc > best_val_acc:
        best_val_acc = vl_acc
        torch.save(teacher.state_dict(), 'best_teacher.pth')
    
    print(f"  Epoch [{epoch:02d}/{TEACHER_EPOCHS}]  "
          f"Train Loss: {tr_loss:.4f}  Train Acc: {tr_acc*100:.2f}%  "
          f"Val Acc: {vl_acc*100:.2f}%")

print(f"\n✅ Teacher best val accuracy: {best_val_acc*100:.2f}%")

In [ ]:
# ── Teacher Baseline Test Accuracy ───────────────────────────
teacher.load_state_dict(torch.load('best_teacher.pth', map_location=DEVICE))
_, teacher_baseline_acc, t_preds, t_labels = evaluate(
    teacher, test_loader, criterion, DEVICE, use_clip_img=True
)
print("=" * 55)
print(f"🎓 TEACHER BASELINE ACCURACY (Test): {teacher_baseline_acc*100:.2f}%")
print("=" * 55)

## 9. Student Model — MobileNetV2

In [ ]:
class StudentMobileNet(nn.Module):
    """
    MobileNetV2 backbone + custom classification head.
    Also returns intermediate features for KD alignment.
    """
    def __init__(self, num_classes, pretrained=True):
        super().__init__()
        weights = models.MobileNet_V2_Weights.IMAGENET1K_V1 if pretrained else None
        base = models.mobilenet_v2(weights=weights)
        
        # Extract backbone (everything except final classifier)
        self.features   = base.features          # output: [B, 1280, 7, 7]
        self.pool       = nn.AdaptiveAvgPool2d(1)  # → [B, 1280]
        self.embed_dim  = 1280
        
        # Feature projection for KD (align to teacher's CLIP_EMBED_DIM)
        self.proj = nn.Linear(self.embed_dim, CLIP_EMBED_DIM)
        
        # Classification head
        self.classifier = nn.Sequential(
            nn.Dropout(0.2),
            nn.Linear(self.embed_dim, num_classes)
        )
    
    def forward(self, x):
        feats  = self.pool(self.features(x)).flatten(1)   # [B, 1280]
        proj   = self.proj(feats)                          # [B, 768]  — for KD
        logits = self.classifier(feats)                    # [B, num_classes]
        return logits, proj


def make_student(num_classes, pretrained=True):
    return StudentMobileNet(num_classes, pretrained).to(DEVICE)


# Count params
_test_student = make_student(NUM_CLASSES)
total_student = sum(p.numel() for p in _test_student.parameters()) / 1e6
print(f"✅ StudentMobileNet ready.")
print(f"   Total parameters: {total_student:.2f}M")
print(f"   (Teacher CLIP-ViT-L has ~307M params — student is ~{307/total_student:.0f}x smaller)")

### 9.1 Baseline Student Training (WITHOUT Knowledge Distillation)

In [ ]:
STUDENT_EPOCHS = 15

student_baseline = make_student(NUM_CLASSES)
opt_sb = torch.optim.Adam(student_baseline.parameters(), lr=1e-3, weight_decay=1e-4)
sch_sb = torch.optim.lr_scheduler.CosineAnnealingLR(opt_sb, T_max=STUDENT_EPOCHS)

sb_train_losses, sb_val_losses = [], []
sb_train_accs,   sb_val_accs   = [], []

def train_student_baseline(model, loader, optimizer, criterion, device):
    model.train()
    total_loss, correct, total = 0, 0, 0
    for batch in tqdm(loader, leave=False):
        imgs   = batch['mobile_img'].to(device)
        labels = batch['label'].to(device)
        optimizer.zero_grad()
        logits, _ = model(imgs)
        loss = criterion(logits, labels)
        loss.backward()
        optimizer.step()
        total_loss += loss.item() * labels.size(0)
        correct    += (logits.argmax(1) == labels).sum().item()
        total      += labels.size(0)
    return total_loss / total, correct / total

@torch.no_grad()
def eval_student(model, loader, criterion, device):
    model.eval()
    total_loss, correct, total = 0, 0, 0
    all_preds, all_labels = [], []
    for batch in loader:
        imgs   = batch['mobile_img'].to(device)
        labels = batch['label'].to(device)
        logits, _ = model(imgs)
        loss = criterion(logits, labels)
        total_loss += loss.item() * labels.size(0)
        preds = logits.argmax(1)
        correct    += (preds == labels).sum().item()
        total      += labels.size(0)
        all_preds.extend(preds.cpu().tolist())
        all_labels.extend(labels.cpu().tolist())
    return total_loss / total, correct / total, all_preds, all_labels


print("📱 Training Student Baseline (no KD) ...")
best_sb_acc = 0

for epoch in range(1, STUDENT_EPOCHS + 1):
    tr_loss, tr_acc = train_student_baseline(student_baseline, train_loader, opt_sb, criterion, DEVICE)
    vl_loss, vl_acc, _, _ = eval_student(student_baseline, val_loader, criterion, DEVICE)
    sch_sb.step()
    
    sb_train_losses.append(tr_loss)
    sb_val_losses.append(vl_loss)
    sb_train_accs.append(tr_acc)
    sb_val_accs.append(vl_acc)
    
    if vl_acc > best_sb_acc:
        best_sb_acc = vl_acc
        torch.save(student_baseline.state_dict(), 'best_student_baseline.pth')
    
    print(f"  Epoch [{epoch:02d}/{STUDENT_EPOCHS}]  "
          f"Train Loss: {tr_loss:.4f}  Train Acc: {tr_acc*100:.2f}%  "
          f"Val Acc: {vl_acc*100:.2f}%")

print(f"\n✅ Student Baseline best val accuracy: {best_sb_acc*100:.2f}%")

In [ ]:
# Student Baseline Test Accuracy
student_baseline.load_state_dict(torch.load('best_student_baseline.pth', map_location=DEVICE))
_, student_baseline_acc, sb_preds, sb_labels = eval_student(
    student_baseline, test_loader, criterion, DEVICE
)
print("=" * 55)
print(f"📱 STUDENT BASELINE ACCURACY (Test, no KD): {student_baseline_acc*100:.2f}%")
print("=" * 55)

## 10. Knowledge Distillation Training

In [ ]:
# ─────────────────────────────────────────────────────────────
# KD Loss = α * CrossEntropy(student_logits, hard_labels)
#         + β * KLDiv(student_soft, teacher_soft)   [response KD]
#         + γ * MSE(student_proj, teacher_feats)    [feature KD]
# ─────────────────────────────────────────────────────────────

class KDLoss(nn.Module):
    """Combined response-based + feature-based Knowledge Distillation loss."""
    def __init__(self, temperature=4.0, alpha=0.3, beta=0.5, gamma=0.2):
        super().__init__()
        self.T     = temperature
        self.alpha = alpha   # weight for hard-label CE
        self.beta  = beta    # weight for soft-label KL
        self.gamma = gamma   # weight for feature MSE
        self.ce    = nn.CrossEntropyLoss()
        self.kl    = nn.KLDivLoss(reduction='batchmean')
        self.mse   = nn.MSELoss()
    
    def forward(self, s_logits, t_logits, s_proj, t_feats, labels):
        # Hard label loss
        loss_ce = self.ce(s_logits, labels)
        
        # Soft label (response-based KD)
        s_soft = F.log_softmax(s_logits / self.T, dim=1)
        t_soft = F.softmax(t_logits / self.T, dim=1)
        loss_kl = self.kl(s_soft, t_soft) * (self.T ** 2)
        
        # Feature alignment (feature-based KD)
        loss_feat = self.mse(
            F.normalize(s_proj, dim=1),
            F.normalize(t_feats.detach(), dim=1)
        )
        
        total = self.alpha * loss_ce + self.beta * loss_kl + self.gamma * loss_feat
        return total, loss_ce.item(), loss_kl.item(), loss_feat.item()


kd_criterion = KDLoss(temperature=4.0, alpha=0.3, beta=0.5, gamma=0.2)
print("✅ KD Loss ready")
print("   Components: α·CE + β·KL_div + γ·MSE_feature")
print("   Hyperparams: T=4.0, α=0.3, β=0.5, γ=0.2")

In [ ]:
KD_EPOCHS = 10

student_kd = make_student(NUM_CLASSES)
opt_kd = torch.optim.AdamW(student_kd.parameters(), lr=5e-4, weight_decay=1e-4)
sch_kd = torch.optim.lr_scheduler.OneCycleLR(
    opt_kd, max_lr=5e-4, epochs=KD_EPOCHS, steps_per_epoch=len(train_loader)
)

kd_train_losses, kd_val_losses   = [], []
kd_train_accs,   kd_val_accs     = [], []
kd_ce_losses, kd_kl_losses, kd_feat_losses = [], [], []

teacher.eval()  # Teacher stays frozen


def train_kd_epoch(student, teacher, loader, optimizer, scheduler, kd_loss_fn, device):
    student.train()
    total_loss, correct, total = 0, 0, 0
    epoch_ce, epoch_kl, epoch_feat = 0, 0, 0
    
    for batch in tqdm(loader, leave=False):
        clip_imgs   = batch['clip_img'].to(device)
        mobile_imgs = batch['mobile_img'].to(device)
        labels      = batch['label'].to(device)
        
        # Teacher forward (no grad)
        with torch.no_grad():
            t_logits, t_feats = teacher(clip_imgs)
        
        # Student forward
        optimizer.zero_grad()
        s_logits, s_proj = student(mobile_imgs)
        
        # KD loss
        loss, l_ce, l_kl, l_feat = kd_loss_fn(s_logits, t_logits, s_proj, t_feats, labels)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(student.parameters(), max_norm=1.0)
        optimizer.step()
        scheduler.step()
        
        total_loss += loss.item() * labels.size(0)
        correct    += (s_logits.argmax(1) == labels).sum().item()
        total      += labels.size(0)
        epoch_ce   += l_ce;  epoch_kl += l_kl;  epoch_feat += l_feat
    
    n = len(loader)
    return (total_loss / total, correct / total,
            epoch_ce/n, epoch_kl/n, epoch_feat/n)


print("🔄 Knowledge Distillation Training ...")
best_kd_acc = 0

for epoch in range(1, KD_EPOCHS + 1):
    tr_loss, tr_acc, ce_l, kl_l, feat_l = train_kd_epoch(
        student_kd, teacher, train_loader, opt_kd, sch_kd, kd_criterion, DEVICE
    )
    vl_loss, vl_acc, _, _ = eval_student(student_kd, val_loader, criterion, DEVICE)
    
    kd_train_losses.append(tr_loss); kd_val_losses.append(vl_loss)
    kd_train_accs.append(tr_acc);   kd_val_accs.append(vl_acc)
    kd_ce_losses.append(ce_l);      kd_kl_losses.append(kl_l);  kd_feat_losses.append(feat_l)
    
    if vl_acc > best_kd_acc:
        best_kd_acc = vl_acc
        torch.save(student_kd.state_dict(), 'best_student_kd.pth')
    
    print(f"  Epoch [{epoch:02d}/{KD_EPOCHS}]  "
          f"Train Acc: {tr_acc*100:.2f}%  Val Acc: {vl_acc*100:.2f}%  "
          f"[CE={ce_l:.3f}  KL={kl_l:.3f}  Feat={feat_l:.3f}]")

print(f"\n✅ Student KD best val accuracy: {best_kd_acc*100:.2f}%")

In [ ]:
# Post-KD Student Test Accuracy
student_kd.load_state_dict(torch.load('best_student_kd.pth', map_location=DEVICE))
_, student_kd_acc, kd_preds, kd_labels = eval_student(
    student_kd, test_loader, criterion, DEVICE
)
print("=" * 55)
print(f"🔄 STUDENT POST-KD ACCURACY (Test): {student_kd_acc*100:.2f}%")
print("=" * 55)

## 11. Results Comparison & Visualization

In [ ]:
# ── Summary Table ─────────────────────────────────────────────
print("\n" + "═" * 62)
print("          📊 FINAL ACCURACY COMPARISON (Test Set)")
print("═" * 62)
print(f"  {'Model':<38} {'Test Acc':>10}")
print("─" * 62)
print(f"  {'Teacher  — CLIP-ViT-L/14  (~307M params)':<38} {teacher_baseline_acc*100:>9.2f}%")
print(f"  {'Student  — MobileNetV2     (~3.4M params)':<38} {student_baseline_acc*100:>9.2f}%  (no KD)")
print(f"  {'Student  — MobileNetV2 + KD':<38} {student_kd_acc*100:>9.2f}%  ✨")
print("═" * 62)
gain = student_kd_acc - student_baseline_acc
gap  = teacher_baseline_acc - student_kd_acc
print(f"  KD improvement over baseline student : +{gain*100:.2f}%")
print(f"  Remaining gap to teacher             :  {gap*100:.2f}%")
print("═" * 62)

In [ ]:
# ── Training Curves ───────────────────────────────────────────
fig, axes = plt.subplots(2, 3, figsize=(21, 10))

# Teacher training
ep_t = range(1, TEACHER_EPOCHS + 1)
axes[0][0].plot(ep_t, [a*100 for a in teacher_train_accs], 'b-o', label='Train', ms=4)
axes[0][0].plot(ep_t, [a*100 for a in teacher_val_accs],   'b--s', label='Val',   ms=4)
axes[0][0].set_title('🎓 Teacher (CLIP-ViT-L) — Accuracy', fontweight='bold')
axes[0][0].set_xlabel('Epoch'); axes[0][0].set_ylabel('Accuracy (%)')
axes[0][0].legend(); axes[0][0].grid(True, alpha=0.3)

axes[0][1].plot(ep_t, teacher_train_losses, 'b-o', label='Train', ms=4)
axes[0][1].plot(ep_t, teacher_val_losses,   'b--s', label='Val',   ms=4)
axes[0][1].set_title('🎓 Teacher (CLIP-ViT-L) — Loss', fontweight='bold')
axes[0][1].set_xlabel('Epoch'); axes[0][1].set_ylabel('Loss')
axes[0][1].legend(); axes[0][1].grid(True, alpha=0.3)

# KD component losses
ep_kd = range(1, KD_EPOCHS + 1)
axes[0][2].plot(ep_kd, kd_ce_losses,   'r-',  label='CE Loss')
axes[0][2].plot(ep_kd, kd_kl_losses,   'g-',  label='KL (Soft) Loss')
axes[0][2].plot(ep_kd, kd_feat_losses, 'm-',  label='Feature MSE Loss')
axes[0][2].set_title('🔄 KD — Loss Components', fontweight='bold')
axes[0][2].set_xlabel('Epoch'); axes[0][2].set_ylabel('Loss')
axes[0][2].legend(); axes[0][2].grid(True, alpha=0.3)

# Student baseline vs KD accuracy
ep_s = range(1, STUDENT_EPOCHS + 1)
axes[1][0].plot(ep_s, [a*100 for a in sb_val_accs], 'orange', label='Baseline (no KD)', lw=2)
ep_k = range(1, KD_EPOCHS + 1)
axes[1][0].plot(ep_k, [a*100 for a in kd_val_accs], 'green',  label='After KD',         lw=2)
axes[1][0].axhline(y=teacher_baseline_acc*100, color='blue', ls='--', alpha=0.7, label='Teacher Acc')
axes[1][0].set_title('📱 Student Val Accuracy — Baseline vs KD', fontweight='bold')
axes[1][0].set_xlabel('Epoch'); axes[1][0].set_ylabel('Val Accuracy (%)')
axes[1][0].legend(); axes[1][0].grid(True, alpha=0.3)

# Bar chart comparison
models_names = ['Teacher\n(CLIP-ViT-L)\n~307M', 'Student\n(MobileNetV2)\nNo KD', 'Student\n(MobileNetV2)\n+ KD']
accs = [teacher_baseline_acc*100, student_baseline_acc*100, student_kd_acc*100]
bar_colors = ['#4e9af1', '#f18b4e', '#5cb85c']
bars = axes[1][1].bar(models_names, accs, color=bar_colors, edgecolor='white', linewidth=1.5, width=0.5)
for bar, acc in zip(bars, accs):
    axes[1][1].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.5,
                    f'{acc:.1f}%', ha='center', va='bottom', fontweight='bold', fontsize=11)
axes[1][1].set_title('🏆 Test Accuracy Comparison', fontweight='bold')
axes[1][1].set_ylabel('Test Accuracy (%)')
axes[1][1].set_ylim(0, max(accs) * 1.15)
axes[1][1].grid(True, alpha=0.3, axis='y')

# KD improvement annotations
axes[1][2].axis('off')
summary_text = (
    f"Knowledge Distillation Summary\n"
    f"{'━'*38}\n\n"
    f"  Teacher  (CLIP-ViT-L/14)\n"
    f"    Params : ~307M\n"
    f"    Test Acc: {teacher_baseline_acc*100:.2f}%\n\n"
    f"  Student Baseline  (MobileNetV2)\n"
    f"    Params : ~3.4M   (90× smaller)\n"
    f"    Test Acc: {student_baseline_acc*100:.2f}%\n\n"
    f"  Student + KD  (MobileNetV2)\n"
    f"    Params : ~3.4M   (90× smaller)\n"
    f"    Test Acc: {student_kd_acc*100:.2f}%\n\n"
    f"  {'━'*38}\n"
    f"  KD Gain  : +{(student_kd_acc-student_baseline_acc)*100:.2f}%\n"
    f"  Gap to Teacher: {(teacher_baseline_acc-student_kd_acc)*100:.2f}%\n"
)
axes[1][2].text(0.05, 0.95, summary_text, transform=axes[1][2].transAxes,
                fontsize=11, va='top', fontfamily='monospace',
                bbox=dict(boxstyle='round', facecolor='#f0f8ff', alpha=0.9))

plt.suptitle('VQA-RAD Knowledge Distillation: CLIP-ViT-L → MobileNetV2', 
             fontsize=15, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig('kd_results.png', dpi=120, bbox_inches='tight')
plt.show()
print("✅ Saved: kd_results.png")

## 12. Confusion Matrix & Classification Report

In [ ]:
def plot_confusion_matrix(preds, labels, le, title, ax, top_n=20):
    """Plot normalized confusion matrix for top-N classes."""
    class_names = le.classes_
    # Filter to top-N most frequent classes
    counts = Counter(labels)
    top_classes = [c for c, _ in counts.most_common(top_n)]
    mask = [i for i, l in enumerate(labels) if l in top_classes]
    
    p_filt = [preds[i]  for i in mask]
    l_filt = [labels[i] for i in mask]
    class_ids = sorted(set(l_filt))
    class_lbls = [class_names[c] for c in class_ids]
    
    cm = confusion_matrix(l_filt, p_filt, labels=class_ids, normalize='true')
    sns.heatmap(cm, ax=ax, xticklabels=class_lbls, yticklabels=class_lbls,
                cmap='Blues', vmin=0, vmax=1, annot=len(class_ids)<=15,
                fmt='.2f', linewidths=0.3)
    ax.set_title(title, fontweight='bold', fontsize=11)
    ax.set_xlabel('Predicted'); ax.set_ylabel('True')
    ax.tick_params(axis='x', rotation=45, labelsize=7)
    ax.tick_params(axis='y', rotation=0,  labelsize=7)


fig, axes = plt.subplots(1, 3, figsize=(24, 8))

plot_confusion_matrix(t_preds,  t_labels,  le, f'Teacher (CLIP-ViT-L)\nAcc={teacher_baseline_acc*100:.1f}%',   axes[0])
plot_confusion_matrix(sb_preds, sb_labels, le, f'Student Baseline\nAcc={student_baseline_acc*100:.1f}%',       axes[1])
plot_confusion_matrix(kd_preds, kd_labels, le, f'Student + KD\nAcc={student_kd_acc*100:.1f}%',                 axes[2])

plt.suptitle('Confusion Matrices — Top-20 Answer Classes (Normalized)', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('confusion_matrices.png', dpi=120, bbox_inches='tight')
plt.show()

In [ ]:
# Per-class report for KD Student
print("📋 Classification Report — Student + KD (top-20 classes):")
counts = Counter(kd_labels)
top_20 = [c for c, _ in counts.most_common(20)]
mask   = [i for i, l in enumerate(kd_labels) if l in top_20]
p_filt = [kd_preds[i]  for i in mask]
l_filt = [kd_labels[i] for i in mask]
target_names = [le.classes_[c] for c in sorted(set(l_filt))]
print(classification_report(l_filt, p_filt, target_names=target_names, labels=sorted(set(l_filt))))

## 13. Sample Predictions — Visual Comparison

In [ ]:
@torch.no_grad()
def get_predictions_sample(teacher, student_bl, student_kd, dataset, n=12, device=DEVICE):
    teacher.eval(); student_bl.eval(); student_kd.eval()
    indices = random.sample(range(len(dataset)), n)
    results = []
    
    for idx in indices:
        item = dataset[idx]
        clip_img   = item['clip_img'].unsqueeze(0).to(device)
        mobile_img = item['mobile_img'].unsqueeze(0).to(device)
        label      = item['label'].item()
        
        t_logits, _ = teacher(clip_img)
        sb_logits, _ = student_bl(mobile_img)
        kd_logits, _ = student_kd(mobile_img)
        
        results.append({
            'image_path': dataset.df.iloc[idx]['image_path'],
            'question':   item['question'],
            'true_answer': item['answer'],
            'gt_label':   label,
            'teacher_pred': le.classes_[t_logits.argmax(1).item()],
            'baseline_pred': le.classes_[sb_logits.argmax(1).item()],
            'kd_pred':     le.classes_[kd_logits.argmax(1).item()],
        })
    return results


samples = get_predictions_sample(teacher, student_baseline, student_kd, test_ds, n=12)

n_cols = 4
n_rows = (len(samples) + n_cols - 1) // n_cols
fig, axes = plt.subplots(n_rows, n_cols, figsize=(22, n_rows * 6))
axes = axes.flatten()

for i, s in enumerate(samples):
    ax = axes[i]
    try:
        img = Image.open(s['image_path']).convert('RGB')
        ax.imshow(img)
    except:
        ax.set_facecolor('#111')
    
    gt = s['true_answer']
    t  = s['teacher_pred']
    b  = s['baseline_pred']
    k  = s['kd_pred']
    
    t_ok  = '✅' if t == gt else '❌'
    b_ok  = '✅' if b == gt else '❌'
    k_ok  = '✅' if k == gt else '❌'
    
    q_short = s['question'][:55] + ('…' if len(s['question'])>55 else '')
    title = (
        f"Q: {q_short}\n"
        f"GT: {gt}\n"
        f"Teacher: {t} {t_ok}\n"
        f"Student (no KD): {b} {b_ok}\n"
        f"Student (KD): {k} {k_ok}"
    )
    ax.set_title(title, fontsize=8, pad=3,
                 bbox=dict(boxstyle='round', facecolor='lightyellow', alpha=0.85))
    ax.axis('off')

for j in range(len(samples), len(axes)):
    axes[j].axis('off')

plt.suptitle('Sample Predictions: Teacher vs Student Baseline vs Student+KD',
             fontsize=14, fontweight='bold', y=1.01)
plt.tight_layout()
plt.savefig('sample_predictions.png', dpi=120, bbox_inches='tight')
plt.show()
print("✅ Saved: sample_predictions.png")

## 14. Feature Space Visualization (t-SNE)

In [ ]:
from sklearn.manifold import TSNE

@torch.no_grad()
def extract_features(model, loader, device, use_clip=False, max_batches=15):
    model.eval()
    feats_all, labels_all = [], []
    for i, batch in enumerate(loader):
        if i >= max_batches:
            break
        imgs   = batch['clip_img' if use_clip else 'mobile_img'].to(device)
        labels = batch['label']
        output = model(imgs)
        feats  = output[1] if isinstance(output, tuple) else output
        feats_all.append(feats.cpu().float().numpy())
        labels_all.append(labels.numpy())
    return np.concatenate(feats_all), np.concatenate(labels_all)


print("⏳ Extracting features for t-SNE ...")
t_feats_np,  t_lbls_np  = extract_features(teacher,          test_loader, DEVICE, use_clip=True)
sb_feats_np, sb_lbls_np = extract_features(student_baseline, test_loader, DEVICE, use_clip=False)
kd_feats_np, kd_lbls_np = extract_features(student_kd,       test_loader, DEVICE, use_clip=False)

print("⏳ Running t-SNE ...")
def run_tsne(feats):
    return TSNE(n_components=2, perplexity=30, random_state=SEED,
                n_iter=500, learning_rate='auto', init='pca').fit_transform(feats)

t_2d  = run_tsne(t_feats_np)
sb_2d = run_tsne(sb_feats_np)
kd_2d = run_tsne(kd_feats_np)

TOP_TSNE = 10   # Only show top-10 classes for clarity
top_10_classes = [c for c, _ in Counter(t_lbls_np.tolist()).most_common(TOP_TSNE)]

def plot_tsne(embed, labels, classes, ax, title):
    mask = np.isin(labels, classes)
    cmap = plt.cm.get_cmap('tab10', len(classes))
    class_map = {c: i for i, c in enumerate(sorted(classes))}
    for j, c in enumerate(sorted(classes)):
        m = (labels == c) & mask
        if m.sum() == 0:
            continue
        ax.scatter(embed[m, 0], embed[m, 1], s=12, alpha=0.65,
                   color=cmap(j), label=le.classes_[c])
    ax.set_title(title, fontweight='bold', fontsize=11)
    ax.set_xticks([]); ax.set_yticks([])
    ax.legend(fontsize=6, markerscale=2, bbox_to_anchor=(1.01, 1), loc='upper left')

fig, axes = plt.subplots(1, 3, figsize=(21, 6))
plot_tsne(t_2d,  t_lbls_np,  top_10_classes, axes[0], f'Teacher Features (CLIP-ViT-L)\nAcc={teacher_baseline_acc*100:.1f}%')
plot_tsne(sb_2d, sb_lbls_np, top_10_classes, axes[1], f'Student Features (No KD)\nAcc={student_baseline_acc*100:.1f}%')
plot_tsne(kd_2d, kd_lbls_np, top_10_classes, axes[2], f'Student Features (After KD)\nAcc={student_kd_acc*100:.1f}%')

plt.suptitle('t-SNE Feature Space — Top-10 Answer Classes', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('tsne_features.png', dpi=120, bbox_inches='tight')
plt.show()
print("✅ Saved: tsne_features.png")

## 15. Final Summary Dashboard

In [ ]:
fig = plt.figure(figsize=(18, 10))
fig.patch.set_facecolor('#0d1117')

# ── Title ────────────────────────────────────────────────────
fig.text(0.5, 0.95, 'VQA-RAD Knowledge Distillation — Final Summary Dashboard',
         ha='center', va='top', fontsize=17, fontweight='bold', color='white')
fig.text(0.5, 0.91, 'Teacher: CLIP-ViT-L/14  →  Student: MobileNetV2',
         ha='center', va='top', fontsize=12, color='#8ecae6')

# ── Accuracy Bar ─────────────────────────────────────────────
ax1 = fig.add_axes([0.05, 0.54, 0.28, 0.32])
ax1.set_facecolor('#161b22')
bars = ax1.bar(
    ['Teacher\nCLIP-ViT-L', 'Student\nBaseline', 'Student\n+KD'],
    [teacher_baseline_acc*100, student_baseline_acc*100, student_kd_acc*100],
    color=['#4e9af1','#f18b4e','#5cb85c'], edgecolor='white', linewidth=1.2
)
for bar, acc in zip(bars, [teacher_baseline_acc, student_baseline_acc, student_kd_acc]):
    ax1.text(bar.get_x()+bar.get_width()/2, bar.get_height()+0.5,
             f'{acc*100:.1f}%', ha='center', va='bottom', fontweight='bold',
             fontsize=10, color='white')
ax1.set_title('Test Accuracy', color='white', fontweight='bold')
ax1.set_ylabel('Accuracy (%)', color='white')
ax1.tick_params(colors='white'); ax1.spines[:].set_color('#444')
ax1.set_facecolor('#161b22'); ax1.yaxis.label.set_color('white')
ax1.set_ylim(0, max(teacher_baseline_acc*100, 100) * 1.12)

# ── KD Gain Over Epochs ───────────────────────────────────────
ax2 = fig.add_axes([0.38, 0.54, 0.28, 0.32])
ax2.set_facecolor('#161b22')
ep_k = range(1, KD_EPOCHS+1)
ax2.plot(ep_k, [a*100 for a in kd_val_accs],   '#5cb85c', lw=2, label='KD Student')
ax2.plot(range(1, STUDENT_EPOCHS+1), [a*100 for a in sb_val_accs], '#f18b4e', lw=2, ls='--', label='Baseline Student')
ax2.axhline(teacher_baseline_acc*100, color='#4e9af1', ls=':', lw=1.5, label='Teacher')
ax2.set_title('Validation Accuracy over Epochs', color='white', fontweight='bold')
ax2.set_xlabel('Epoch', color='white')
ax2.set_ylabel('Val Accuracy (%)', color='white')
ax2.tick_params(colors='white'); ax2.spines[:].set_color('#444')
ax2.legend(fontsize=8, facecolor='#2d333b', labelcolor='white')
ax2.set_facecolor('#161b22')

# ── KD Loss Components ────────────────────────────────────────
ax3 = fig.add_axes([0.71, 0.54, 0.26, 0.32])
ax3.set_facecolor('#161b22')
ax3.plot(ep_k, kd_ce_losses,   '#f18b4e', lw=2, label='CE (Hard Labels)')
ax3.plot(ep_k, kd_kl_losses,   '#5cb85c', lw=2, label='KL (Soft Labels)')
ax3.plot(ep_k, kd_feat_losses, '#b388ff', lw=2, label='Feature MSE')
ax3.set_title('KD Loss Components', color='white', fontweight='bold')
ax3.set_xlabel('Epoch', color='white')
ax3.set_ylabel('Loss', color='white')
ax3.tick_params(colors='white'); ax3.spines[:].set_color('#444')
ax3.legend(fontsize=8, facecolor='#2d333b', labelcolor='white')
ax3.set_facecolor('#161b22')

# ── Stats Table ───────────────────────────────────────────────
ax4 = fig.add_axes([0.05, 0.06, 0.90, 0.40])
ax4.set_facecolor('#161b22')
ax4.axis('off')

table_data = [
    ['Model', 'Params', 'Backbone', 'Train Strategy', 'Test Accuracy', 'Size vs Teacher'],
    ['CLIP-ViT-L/14 (Teacher)', '~307M', 'Vision Transformer L/14', 'Classifier Head Fine-tune', f'{teacher_baseline_acc*100:.2f}%', '1×'],
    ['MobileNetV2 (Baseline)', '~3.4M', 'Inverted Residuals', 'Standard CE Training', f'{student_baseline_acc*100:.2f}%', '~90× smaller'],
    ['MobileNetV2 + KD (Ours)', '~3.4M', 'Inverted Residuals', 'KD: CE + KL-Div + FeatMSE', f'{student_kd_acc*100:.2f}%', '~90× smaller'],
]

tbl = ax4.table(
    cellText=table_data[1:],
    colLabels=table_data[0],
    loc='center',
    cellLoc='center'
)
tbl.auto_set_font_size(False)
tbl.set_fontsize(10)
tbl.scale(1.0, 2.5)

header_color = '#1f6feb'
row_colors = ['#21262d', '#2d333b', '#161b22']

for (row, col), cell in tbl.get_celld().items():
    cell.set_edgecolor('#444')
    if row == 0:
        cell.set_facecolor(header_color)
        cell.set_text_props(color='white', fontweight='bold')
    else:
        cell.set_facecolor(row_colors[(row-1) % len(row_colors)])
        cell.set_text_props(color='white')
    if col == 4 and row > 0:
        accs_col = [teacher_baseline_acc, student_baseline_acc, student_kd_acc]
        if row-1 == 2:
            cell.set_facecolor('#1a472a')  # highlight KD student

plt.savefig('final_dashboard.png', dpi=130, bbox_inches='tight', facecolor='#0d1117')
plt.show()
print("✅ Saved: final_dashboard.png")

## 16. Save All Models & Export Results

In [ ]:
# Save final models
torch.save(teacher.state_dict(),        'teacher_clip_vitl.pth')
torch.save(student_baseline.state_dict(),'student_mobilenet_baseline.pth')
torch.save(student_kd.state_dict(),     'student_mobilenet_kd.pth')

# Save label encoder
import pickle
with open('label_encoder.pkl', 'wb') as f:
    pickle.dump(le, f)

# Save results as CSV
results_df = pd.DataFrame({
    'model':      ['Teacher (CLIP-ViT-L)', 'Student Baseline (MobileNetV2)', 'Student + KD (MobileNetV2)'],
    'params_M':   [307, 3.4, 3.4],
    'test_acc':   [teacher_baseline_acc, student_baseline_acc, student_kd_acc],
    'test_acc_pct': [teacher_baseline_acc*100, student_baseline_acc*100, student_kd_acc*100]
})
results_df.to_csv('kd_results.csv', index=False)

print("✅ Saved artifacts:")
print("   teacher_clip_vitl.pth")
print("   student_mobilenet_baseline.pth")
print("   student_mobilenet_kd.pth")
print("   label_encoder.pkl")
print("   kd_results.csv")
print("   vqa_rad_samples.png")
print("   kd_results.png")
print("   confusion_matrices.png")
print("   sample_predictions.png")
print("   tsne_features.png")
print("   final_dashboard.png")

print("\n" + "═" * 55)
print("          🎉 KNOWLEDGE DISTILLATION COMPLETE")
print("═" * 55)
print(f"  Teacher (CLIP-ViT-L/14)   : {teacher_baseline_acc*100:.2f}%")
print(f"  Student Baseline           : {student_baseline_acc*100:.2f}%")
print(f"  Student + KD               : {student_kd_acc*100:.2f}%  (+{(student_kd_acc-student_baseline_acc)*100:.2f}% gain)")
print("═" * 55)